# Técnicas de Diseño y Validación de Modelos · Universidad Blas Pascal
## Módulo 1 · Diseño del proceso de modelado
## Laboratorio: Brecha de generalización en modelos polinómicos

<div style="display:flex; justify-content:flex-start; margin: 8px 0 18px 0;">
  <a href="https://colab.research.google.com/github/GabrielaOjcius/Laboratorios-Tecnicas-Diseno-Validacion-Modelos/blob/main/Laboratorios%20opcionales/M1_Notebook.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/>
  </a>
  <span style="font-size: 0.98em;">En Colab, hac&eacute; una copia en tu Drive antes de editar.</span>
</div>


---

### Objetivo
Observar empíricamente el fenómeno central del Módulo 1: el error de entrenamiento decrece monotónicamente con la complejidad del modelo, mientras que el error de validación describe una curva en U. Este experimento permite cuantificar la brecha de generalización y vincularla con el compromiso sesgo-varianza.

### Instrucciones
1. Ejecutar las celdas de código en orden (Entorno de ejecución → Ejecutar todo).
2. Completar las **celdas de respuesta** marcadas con `# ✏️ RESPONDA AQUÍ`.
3. Redactar los análisis en las **celdas Markdown** indicadas.
4. No modificar el código de las celdas de setup (B.1) sin justificación.

In [ ]:
# ─── B.1. Importaciones y generación de datos sintéticos ──────────────────────
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, learning_curve, KFold, cross_val_score
from sklearn.metrics import mean_squared_error

# Semilla fija para reproducibilidad
RNG = np.random.RandomState(42)

# Función subyacente conocida: nos permite medir el error irreducible
def f_real(x):
    return np.sin(2 * np.pi * x)

# Generación de datos: señal + ruido gaussiano
N = 80
SIGMA = 0.20  # desviación estándar del ruido
X = np.sort(RNG.rand(N))
y = f_real(X) + RNG.normal(0, SIGMA, N)

# Partición 80 % entrenamiento / 20 % validación
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.20, random_state=42)

print(f'Tamaño total: {N}  |  Train: {len(y_tr)}  |  Validación: {len(y_va)}')
print(f'Error irreducible teórico (σ²): {SIGMA**2:.4f}')

In [ ]:
# ─── B.2. Ajuste con distinta complejidad: grados {1, 3, 9, 15} ───────────────
GRADOS = [1, 3, 9, 15]
mse_tr, mse_va, modelos = [], [], {}

for g in GRADOS:
    pipe = Pipeline([
        ('poly',   PolynomialFeatures(degree=g, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg',    LinearRegression()),
    ])
    pipe.fit(X_tr[:, None], y_tr)
    modelos[g] = pipe
    mse_tr.append(mean_squared_error(y_tr, pipe.predict(X_tr[:, None])))
    mse_va.append(mean_squared_error(y_va, pipe.predict(X_va[:, None])))

# ── Tabla de resultados ───────────────────────────────────────────────────────
tabla = pd.DataFrame({
    'Grado': GRADOS,
    'MSE entrenamiento': np.round(mse_tr, 4),
    'MSE validación':    np.round(mse_va, 4),
    'Brecha (val−train)':np.round(np.array(mse_va) - np.array(mse_tr), 4),
})
print(tabla.to_string(index=False))

In [ ]:
# ─── B.3. Visualización de los ajustes ───────────────────────────────────────
X_plot = np.linspace(0, 1, 400)
titulos = ['Grado 1\n(¿subajuste?)', 'Grado 3\n(¿buen ajuste?)',
           'Grado 9\n(¿sobreajuste?)', 'Grado 15\n(¿sobreajuste severo?)']

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharey=True)
fig.suptitle('Ajuste polinómico con distinta complejidad', fontsize=13, fontweight='bold')

for ax, g, titulo in zip(axes, GRADOS, titulos):
    ax.plot(X_plot, f_real(X_plot), '--', color='gray', lw=1.5, label='f real')
    ax.plot(X_plot, modelos[g].predict(X_plot[:, None]), color='#8B1A2F', lw=2, label='Modelo')
    ax.scatter(X_tr, y_tr, s=20, alpha=0.7, color='steelblue', label='Train')
    ax.scatter(X_va, y_va, s=20, alpha=0.7, color='#2CA02C', label='Val', marker='^')
    ax.set_title(titulo, fontsize=11)
    ax.set_ylim(-2.0, 2.0)
    ax.set_xlim(0, 1)
    ax.set_xlabel('x')

axes[0].set_ylabel('y'); axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ─── B.4. Curva de generalización (error vs. complejidad) ──────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(GRADOS, mse_tr, '-o', color='steelblue', lw=2, markersize=7, label='MSE entrenamiento')
ax.plot(GRADOS, mse_va, '-o', color='#8B1A2F',  lw=2, markersize=7, label='MSE validación')
ax.axhline(SIGMA**2, linestyle=':', color='gray', lw=1.5,
           label=f'Error irreducible σ²={SIGMA**2:.3f}')

ax.fill_between(GRADOS, mse_tr, mse_va,
                where=[v > t for v, t in zip(mse_va, mse_tr)],
                alpha=0.12, color='#8B1A2F', label='Brecha de generalización')

ax.set_xlabel('Grado del polinomio', fontsize=12)
ax.set_ylabel('MSE', fontsize=12)
ax.set_title('Brecha de generalización en función de la complejidad', fontsize=12, fontweight='bold')
ax.set_xticks(GRADOS)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('\nGrado con menor MSE de validación:', GRADOS[np.argmin(mse_va)])

In [ ]:
# ─── B.5. Curvas de aprendizaje: alto sesgo vs. alta varianza ──────────────────
cv = KFold(n_splits=5, shuffle=True, random_state=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
fig.suptitle('Curvas de aprendizaje', fontsize=13, fontweight='bold')

for ax, g, titulo in zip(axes,
                          [1, 8],
                          ['Grado 1 — Alto sesgo (subajuste)', 'Grado 8 — Alta varianza (sobreajuste)']):
    pipe = Pipeline([('poly', PolynomialFeatures(g)), ('reg', LinearRegression())])
    sizes, tr_sc, va_sc = learning_curve(
        pipe, X[:, None], y,
        cv=cv, train_sizes=np.linspace(0.15, 1.0, 8),
        scoring='neg_mean_squared_error', random_state=0
    )
    ax.plot(sizes, -tr_sc.mean(1), '-o', color='steelblue', lw=2, label='Entrenamiento')
    ax.fill_between(sizes, -tr_sc.mean(1)-tr_sc.std(1), -tr_sc.mean(1)+tr_sc.std(1),
                    alpha=0.15, color='steelblue')
    ax.plot(sizes, -va_sc.mean(1), '-o', color='#8B1A2F', lw=2, label='Validación')
    ax.fill_between(sizes, -va_sc.mean(1)-va_sc.std(1), -va_sc.mean(1)+va_sc.std(1),
                    alpha=0.15, color='#8B1A2F')
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel('Tamaño del conjunto de entrenamiento')
    ax.legend(); ax.grid(alpha=0.3)

axes[0].set_ylabel('MSE')
plt.tight_layout()
plt.show()

In [ ]:
# ─── B.6. Pipeline reproducible + validación cruzada ─────────────────────────
pipe_cv = Pipeline([
    ('poly',   PolynomialFeatures(degree=3, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression()),
])

cv5 = KFold(n_splits=5, shuffle=True, random_state=7)
scores = cross_val_score(pipe_cv, X[:, None], y, cv=cv5, scoring='neg_mean_squared_error')

print('MSE por fold:', np.round(-scores, 4))
print(f'MSE medio CV: {-scores.mean():.4f} ± {scores.std():.4f}')
print()
print('Nota: el Pipeline garantiza que el StandardScaler se ajusta')
print('solo con los datos de entrenamiento de cada fold.')

---
## Preguntas de análisis
Responder en las siguientes celdas. Hacer referencia explícita a los valores numéricos observados en la tabla de la celda B.2.

### Pregunta 1
¿En qué grado o grados aparece el subajuste? Justificar con los valores de MSE observados.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 2
¿En qué grado o grados aparece el sobreajuste? Identificar el síntoma cuantitativo.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 3
¿Qué grado seleccionaría como mejor modelo? Justificar con el criterio del menor error de validación.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 4
¿Por qué no es válido seleccionar el grado por el MSE de entrenamiento? Conectar con los conceptos de riesgo empírico y riesgo esperado.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 5
¿Qué cambia si se usa el conjunto de test en lugar del de validación para elegir el grado? Nombrar el riesgo metodológico con el término técnico correcto.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

In [ ]:
# ─── B.7. Exploración adicional: impacto del nivel de ruido ─────────────────
def experimento_ruido(sigma, rng_seed=42):
    rng = np.random.RandomState(rng_seed)
    X_ = np.sort(rng.rand(N))
    y_ = f_real(X_) + rng.normal(0, sigma, N)
    X_tr_, X_va_, y_tr_, y_va_ = train_test_split(X_, y_, test_size=0.2, random_state=42)

    mse_t, mse_v = [], []
    for g in GRADOS:
        pipe_ = Pipeline([
            ('poly',   PolynomialFeatures(degree=g, include_bias=False)),
            ('scaler', StandardScaler()),
            ('reg',    LinearRegression()),
        ])
        pipe_.fit(X_tr_[:, None], y_tr_)
        mse_t.append(mean_squared_error(y_tr_, pipe_.predict(X_tr_[:, None])))
        mse_v.append(mean_squared_error(y_va_, pipe_.predict(X_va_[:, None])))
    return mse_t, mse_v

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
fig.suptitle('Efecto del nivel de ruido sobre la brecha de generalización', fontsize=13, fontweight='bold')

for ax, sig in zip(axes, [0.10, 0.20, 0.40]):
    mt, mv = experimento_ruido(sig)
    ax.plot(GRADOS, mt, '-o', color='steelblue', lw=2, label='Train')
    ax.plot(GRADOS, mv, '-o', color='#8B1A2F',  lw=2, label='Validación')
    ax.axhline(sig**2, linestyle=':', color='gray', lw=1.5, label=f'σ²={sig**2:.3f}')
    ax.set_title(f'σ = {sig}', fontsize=11)
    ax.set_xlabel('Grado'); ax.set_xticks(GRADOS)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

axes[0].set_ylabel('MSE')
plt.tight_layout()
plt.show()

### Exploración adicional
Comparar los tres escenarios de ruido (σ = 0.1, 0.2, 0.4) y responder:

¿En cuál de los tres escenarios el modelo flexible (grado 15) tiene peor desempeño relativo en validación? ¿Por qué la severidad del sobreajuste depende de la relación señal/ruido y no solo de la complejidad del modelo?

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`